# LGA Boundaries — Clean and Insert into DB

This notebook builds a data pipeline for the ABS 2024 Local Government Areas shapefile and writes cleaned data (including full polygon geometry) to PostGIS.

## Cleaning rules
- read the `LGA_2024_AUST_GDA2020.shp` shapefile
- rename ABS-style columns (`LGA_CODE24` etc.) to readable snake_case
- trim and normalize whitespace in all text fields; blanks → NaN
- drop rows with missing or empty geometry (non-spatial pseudo-LGAs like 'No usual address')
- drop rows where `area_sqkm <= 0`
- simplify geometry to ~100m tolerance for fast frontend rendering
- promote all geometries to `MultiPolygon` for type uniformity
- round `area_sqkm` to 4 decimal places
- return a cleaning summary reporting each drop reason and count

## Storage approach
Geometry is stored as WKT (Well-Known Text) in a plain `TEXT` column because PostGIS is not available on this Postgres 18 instance. Boundaries are preserved losslessly and can be read back into GeoPandas via `gpd.GeoSeries.from_wkt`. Spatial queries that would normally run in the database (e.g. `ST_Contains`) are done in Python instead. Geometry is pre-simplified to ~100m tolerance for fast loading on the frontend (Leaflet); if higher precision is ever needed for analysis, re-run this pipeline from the shapefile with a smaller tolerance.

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import pyogrio

INPUT_SHP = Path("../input/LGA_2024_AUST_GDA2020/LGA_2024_AUST_GDA2020.shp")
INPUT_SHP

PosixPath('input/LGA_2024_AUST_GDA2020/LGA_2024_AUST_GDA2020.shp')

In [33]:
# Preview layer info without loading geometries
info = pyogrio.read_info(INPUT_SHP)
print(f"Features : {info['features']:,}")
print(f"CRS      : {info['crs']}")
print(f"Fields   : {list(info['fields'])}")
print(f"Geometry : {info['geometry_type']}")

Features : 566
CRS      : EPSG:7844
Fields   : ['LGA_CODE24', 'LGA_NAME24', 'STE_CODE21', 'STE_NAME21', 'AUS_CODE21', 'AUS_NAME21', 'AREASQKM', 'LOCI_URI21']
Geometry : Polygon


## Estimating data size before insertion

Before pushing 500+ polygons (some with thousands of vertices) into Postgres, it's worth knowing roughly how much space they'll take. Three useful measures:

1. **In-memory size** — how much RAM the GeoDataFrame uses in pandas.
2. **Serialised geometry size** — how many bytes each polygon takes as WKB (Well-Known Binary, the format PostGIS stores). This is the best estimate of on-disk size for the geometry column.
3. **Total estimated row size** — geometry + text attributes + numeric fields + Postgres overhead (~30 bytes/row for headers, TID, etc.).

WKB is more compact than WKT and is what `to_postgis` / `geom_from_wkb` actually transmit, so it's the right thing to measure.

In [34]:
from shapely.geometry import MultiPolygon

RENAME = {
    "LGA_CODE24": "lga_code",
    "LGA_NAME24": "lga_name",
    "STE_CODE21": "state_code",
    "STE_NAME21": "state_name",
    "AREASQKM":   "area_sqkm",
    "LOCI_URI21": "loci_uri",
}

def clean_lga_boundaries(input_path: Path):
    gdf = gpd.read_file(input_path)
    rows_before = len(gdf)
    dropped = {}

    # Drop columns we don't keep (AUS_CODE21, AUS_NAME21 — same value for all rows)
    gdf = gdf.drop(columns=[c for c in ["AUS_CODE21", "AUS_NAME21"] if c in gdf.columns])
    gdf = gdf.rename(columns=RENAME)

    # Clean text columns: normalize whitespace, blanks → NA
    text_cols = ["lga_code", "lga_name", "state_code", "state_name", "loci_uri"]
    for col in text_cols:
        if col in gdf.columns:
            s = gdf[col].astype("string")
            s = s.str.replace(r"\s+", " ", regex=True).str.strip()
            s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
            gdf[col] = s

    # Drop rows with missing or empty geometry
    # (ABS includes pseudo-LGAs like 'No usual address (NSW)' with null geometry)
    mask = gdf.geometry.isna() | gdf.geometry.is_empty
    dropped["missing_geometry"] = int(mask.sum())
    gdf = gdf[~mask].copy()

    # Drop rows where area is zero / negative / missing
    gdf["area_sqkm"] = pd.to_numeric(gdf["area_sqkm"], errors="coerce").round(4)
    mask = gdf["area_sqkm"].isna() | (gdf["area_sqkm"] <= 0)
    dropped["invalid_area"] = int(mask.sum())
    gdf = gdf[~mask].copy()

    # Simplify for frontend rendering.
    # preserve_topology=True keeps polygons valid (no self-intersections) and
    # keeps adjacent LGAs sharing edges correctly.
    gdf["geometry"] = gdf.geometry.simplify(tolerance=0.001, preserve_topology=True)

    # Promote Polygon → MultiPolygon for uniformity (simplify can return Polygon)
    gdf["geometry"] = gdf.geometry.apply(
        lambda g: MultiPolygon([g]) if g.geom_type == "Polygon" else g
    )

    # Reset index and reorder columns
    gdf = gdf.reset_index(drop=True)
    final_cols = [
        "lga_code", "lga_name", "state_code", "state_name",
        "area_sqkm", "loci_uri", "geometry",
    ]
    gdf = gdf[final_cols]

    summary = {
        "input_rows": rows_before,
        "output_rows": len(gdf),
        "total_dropped": rows_before - len(gdf),
        "dropped_by_reason": dropped,
        "column_count": gdf.shape[1],
        "crs": str(gdf.crs),
    }
    return gdf, summary

In [35]:
clean_gdf, summary = clean_lga_boundaries(INPUT_SHP)
summary

{'input_rows': 566,
 'output_rows': 547,
 'total_dropped': 19,
 'dropped_by_reason': {'missing_geometry': 19, 'invalid_area': 0},
 'column_count': 7,
 'crs': 'EPSG:7844'}

In [36]:
clean_gdf.head()

,lga_code,lga_name,state_code,state_name,area_sqkm,loci_uri,geometry
0,10050,Albury,1,New South Wales,305.6386,https://linked.data.gov.au/dataset/asgsed3/LGA...,"MULTIPOLYGON (((146.86566 -36.07292, 146.86409..."
1,10180,Armidale,1,New South Wales,7809.4406,https://linked.data.gov.au/dataset/asgsed3/LGA...,"MULTIPOLYGON (((152.38816 -30.52639, 152.38619..."
2,10250,Ballina,1,New South Wales,484.9692,https://linked.data.gov.au/dataset/asgsed3/LGA...,"MULTIPOLYGON (((153.57117 -28.87275, 153.57198..."
3,10300,Balranald,1,New South Wales,21690.7493,https://linked.data.gov.au/dataset/asgsed3/LGA...,"MULTIPOLYGON (((142.99242 -33.85356, 143.03952..."
4,10470,Bathurst,1,New South Wales,3817.8645,https://linked.data.gov.au/dataset/asgsed3/LGA...,"MULTIPOLYGON (((149.84877 -33.52784, 149.8363 ..."


In [37]:
# Sanity checks
print(f"Missing lga_code: {int(clean_gdf['lga_code'].isna().sum())}")
print(f"Duplicate lga_code: {int(clean_gdf['lga_code'].duplicated().sum())}")
print(f"Rows with area_sqkm <= 0: {int((clean_gdf['area_sqkm'] <= 0).sum())}")
print(f"Invalid geometries: {int((~clean_gdf.geometry.is_valid).sum())}")
print(f"Geometry types: {clean_gdf.geometry.geom_type.value_counts().to_dict()}")
print(f"CRS: {clean_gdf.crs}")
print()
print("State value counts:")
print(clean_gdf['state_name'].value_counts())

Missing lga_code: 0
Duplicate lga_code: 0
Rows with area_sqkm <= 0: 0
Invalid geometries: 0
Geometry types: {'MultiPolygon': 547}
CRS: EPSG:7844

State value counts:
state_name
Western Australia               137
New South Wales                 129
Victoria                         80
Queensland                       78
South Australia                  71
Tasmania                         29
Northern Territory               19
Other Territories                 3
Australian Capital Territory      1
Name: count, dtype: Int64


# Insert Clean Data into DB

In [38]:
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

# -----------------------------
# 1. DATABASE CONNECTION
# -----------------------------
load_dotenv('.env.db', override=True)

DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = os.getenv("DB_PORT")
DB_NAME     = os.getenv("DB_NAME")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
)

engine = create_engine(DATABASE_URL)

# -----------------------------
# 2. CREATE TABLE FROM SCHEMA
# -----------------------------
schema_path = Path("schemas/lga_boundaries.sql")

with open(schema_path, "r", encoding="utf-8") as f:
    schema_sql = f.read()

with engine.begin() as conn:
    conn.execute(text(schema_sql))

print("Schema created or already exists.")
print(DB_NAME)

Schema created or already exists.
carelink


In [39]:
# -----------------------------
# 3. PREPARE DATAFRAME
# -----------------------------
# Convert geometry to WKT text, then drop the geometry column.
# After this point it's a regular DataFrame, not a GeoDataFrame.
df_insert = clean_gdf.copy()
df_insert["geom_wkt"] = clean_gdf.geometry.to_wkt()
df_insert = pd.DataFrame(df_insert.drop(columns=["geometry"]))

target_columns = [
    "lga_code", "lga_name", "state_code", "state_name",
    "area_sqkm", "loci_uri", "geom_wkt",
]
df_insert = df_insert[target_columns]

print(f"Ready to insert {len(df_insert):,} rows.")
print(f"WKT total size: {df_insert['geom_wkt'].str.len().sum() / 1024**2:,.2f} MB")

Ready to insert 547 rows.
WKT total size: 5.42 MB


In [40]:
# -----------------------------
# 4. INSERT INTO DATABASE
# -----------------------------
df_insert.to_sql(
    name="lga_boundaries",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=100,
)

print(f"Inserted {len(df_insert):,} rows into lga_boundaries")

# -----------------------------
# 5. VALIDATE
# -----------------------------
with engine.begin() as conn:
    row_count = conn.execute(text("SELECT COUNT(*) FROM lga_boundaries")).scalar()
    table_size = conn.execute(text(
        "SELECT pg_size_pretty(pg_total_relation_size('lga_boundaries'))"
    )).scalar()
    wkt_size = conn.execute(text(
        "SELECT pg_size_pretty(SUM(pg_column_size(geom_wkt))::bigint) FROM lga_boundaries"
    )).scalar()
    sample = conn.execute(text(
        "SELECT lga_name, LEFT(geom_wkt, 60) || '...' AS wkt_preview FROM lga_boundaries LIMIT 3"
    )).fetchall()

print(f"Rows in table        : {row_count:,}")
print(f"Table size (total)   : {table_size}")
print(f"WKT column size      : {wkt_size}")
print()
for row in sample:
    print(row)

Inserted 547 rows into lga_boundaries
Rows in table        : 547
Table size (total)   : 3760 kB
WKT column size      : 3209 kB

('Albury', 'MULTIPOLYGON (((146.86566 -36.072917, 146.864094 -36.074232,...')
('Armidale', 'MULTIPOLYGON (((152.388162 -30.526395, 152.386194 -30.529817...')
('Ballina', 'MULTIPOLYGON (((153.571174 -28.872745, 153.571982 -28.873584...')


In [41]:
# Spatial sanity check — round-trip WKT back to geometry and run point-in-polygon in Python
from shapely.geometry import Point

df = pd.read_sql(
    "SELECT lga_code, lga_name, state_name, geom_wkt FROM lga_boundaries",
    engine,
)
gdf_back = gpd.GeoDataFrame(
    df,
    geometry=gpd.GeoSeries.from_wkt(df["geom_wkt"]),
    crs="EPSG:7844",
).drop(columns=["geom_wkt"])

melbourne_cbd = Point(144.9631, -37.8136)
hit = gdf_back[gdf_back.geometry.contains(melbourne_cbd)]
print(hit[["lga_code", "lga_name", "state_name", "geometry"]])

    lga_code   lga_name state_name  \
189    24600  Melbourne   Victoria   

                                              geometry  
189  MULTIPOLYGON (((144.97133 -37.82988, 144.96585...  
